# 4.2 VAE Imputation (Joint Pipeline Ready)

Dieses Notebook führt die Inferenz für ein spezifisches VAE-Modell durch. Es ist Teil der gemeinsamen VAE/kNN-Pipeline.

In [ ]:
# ------------------------- Importe -------------------------
import itertools, random
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
import json
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# ------------------------- Injektionen durch Pipeline -------------------------
if 'TARGET_RUN_INDEX' not in globals(): TARGET_RUN_INDEX = 0
if 'INCLUDE_INCOMPLETE' not in globals(): INCLUDE_INCOMPLETE = False
if 'FORCE_MODEL_FOLDER' not in globals(): FORCE_MODEL_FOLDER = None

In [ ]:
# ------------------------- VAE Modell Definition -------------------------
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=16, hidden_dim=512, enc_layers=2, dec_layers=2):
        super(VAE, self).__init__()
        self.leaky_relu = nn.LeakyReLU(0.2)
        
        # ------------------------- Encoder Aufbau -------------------------
        modules_enc = []
        curr = input_dim
        for i in range(enc_layers):
            nxt = hidden_dim // (2**i)
            modules_enc.extend([nn.Linear(curr, nxt), self.leaky_relu])
            curr = nxt
        self.encoder_base = nn.Sequential(*modules_enc)
        
        # ------------------------- Latenter Raum -------------------------
        self.fc_mu = nn.Linear(curr, latent_dim)
        self.fc_logvar = nn.Linear(curr, latent_dim)
        
        # ------------------------- Decoder Aufbau -------------------------
        modules_dec = []
        curr = latent_dim
        for i in range(dec_layers):
            nxt = hidden_dim // (2**(max(0, dec_layers-1-i)))
            modules_dec.extend([nn.Linear(curr, nxt), self.leaky_relu])
            curr = nxt
        self.decoder_base = nn.Sequential(*modules_dec)
        
        # ------------------------- finale Rekonstruktion -------------------------
        self.dec_final = nn.Linear(curr, input_dim)
        
    def encode(self, x): 
        h = self.encoder_base(x)
        return self.fc_mu(h), self.fc_logvar(h)
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std
        
    def decode(self, z): return self.dec_final(self.decoder_base(z))
    
    def forward(self, x):
        mu, lv = self.encode(x)
        z = self.reparameterize(mu, lv)
        return self.decode(z), mu, lv

In [ ]:
# ------------------------- Pfad-Konfiguration & Modelladen -------------------------
base_dir = Path.cwd()
models_root = base_dir.parent / "4.1_VAE_Imputation" / "Models"

if FORCE_MODEL_FOLDER:
    model_dir = models_root / FORCE_MODEL_FOLDER
else:
    timestamp_folders = [f for f in models_root.iterdir() if f.is_dir()]
    model_dir = max(timestamp_folders, key=lambda f: f.stat().st_mtime)

print(f"Using model directory: {model_dir.name}")

# ------------------------- Modell- und Scaler-Dateien finden -------------------------
meta_files = list(model_dir.glob(f"Model_{TARGET_RUN_INDEX:03d}_*_meta.json"))
if not meta_files: raise FileNotFoundError(f"No model found for index {TARGET_RUN_INDEX}")
with open(meta_files[0], 'r') as f: meta = json.load(f)

model_path = list(model_dir.glob(f"Model_{TARGET_RUN_INDEX:03d}_*_vae.pth"))[0]
scaler_path = list(model_dir.glob(f"Model_{TARGET_RUN_INDEX:03d}_*_scaler.joblib"))[0]

features = meta['features_mapped']
vae = VAE(len(features), meta['latent_dim'], meta['hidden_dim'], meta.get('enc_layers',2), meta.get('dec_layers',2))
vae.load_state_dict(torch.load(model_path))
vae.eval()
scaler = joblib.load(scaler_path)

In [ ]:
# ------------------------- Daten laden & Preprocessing -------------------------
preprocessing_root = base_dir.parent.parent / "3_Machine-Learning" / "3.1_Preprocessing" / "Preprocessing"
latest_prep_folder = max([f for f in preprocessing_root.iterdir() if f.is_dir()], key=lambda f: f.stat().st_mtime)
df_full = pd.read_csv(latest_prep_folder / "Preprocessed_SOM_Ready.csv", low_memory=False)

if INCLUDE_INCOMPLETE:
    data_subset = df_full[features].dropna(how='all').copy()
else:
    data_subset = df_full[features].dropna().copy()

X_original_full = data_subset.values
X_train_orig, X_test_orig = train_test_split(X_original_full, test_size=0.1, random_state=42)

# ------------------------- Skalierung & NaN-Handling -------------------------
X_train_scaled = scaler.transform(np.nan_to_num(X_train_orig))
X_train_scaled[np.isnan(X_train_orig)] = np.nan

X_test_scaled = scaler.transform(np.nan_to_num(X_test_orig))
X_test_scaled[np.isnan(X_test_orig)] = np.nan

res_dir = Path("Inference_Results") / model_dir.name
res_dir.mkdir(parents=True, exist_ok=True)

# ------------------------- Inferenz-Loop über Maskierungslevel -------------------------
num_features = len(features)
for k in range(1, num_features):
    print(f"Processing Level {k}...")
    combos = list(itertools.combinations(range(num_features), k))
    
    if len(combos) > 50: combos_to_run = random.sample(combos, 50)
    else: combos_to_run = combos
    
    all_results = []
    
    # ------------------------- Hilfsfunktion für Inferenz (Batching) -------------------------
    def run_inference_on_split(X_orig_split, X_scaled_split, split_name):
        split_results = []
        for combo_indices in combos_to_run:
            X_target_scaled = X_scaled_split.copy()
            # ------------------------- Künstliche Maskierung -------------------------
            is_masked = np.zeros(X_target_scaled.shape, dtype=bool)
            for f_idx in combo_indices:
                mask_condition = ~np.isnan(X_target_scaled[:, f_idx])
                X_target_scaled[mask_condition, f_idx] = 0.0
                is_masked[mask_condition, f_idx] = True
            
            X_target_scaled = np.nan_to_num(X_target_scaled)
            
            # ------------------------- VAE Rekonstruktion -------------------------
            with torch.no_grad():
                recon, _, _ = vae(torch.from_numpy(X_target_scaled).float())
                X_recon_scaled = recon.numpy()
            
            X_recon_orig = scaler.inverse_transform(X_recon_scaled)
            
            # ------------------------- Ergebnisse sammeln -------------------------
            for f_idx in combo_indices:
                mask = is_masked[:, f_idx]
                if not mask.any(): continue
                
                o_vals = X_orig_split[mask, f_idx]
                i_vals = X_recon_orig[mask, f_idx]
                for o, i in zip(o_vals, i_vals):
                    split_results.append({
                        "Feature": features[f_idx],
                        "Original": o,
                        "Imputed": i,
                        "Split": split_name,
                        "Masking_Level": k,
                        "Masked_Combination": str([features[i] for i in combo_indices])
                    })
        return split_results

    # ------------------------- Test-Split (Vollständig) -------------------------
    all_results.extend(run_inference_on_split(X_test_orig, X_test_scaled, "Test"))
    
    # ------------------------- Train-Split (Zusätzlich 10% Stichprobe für Visualisierung) -------------------------
    n_train = len(X_train_orig)
    sample_size = max(1, n_train // 10)
    sample_indices = np.random.choice(n_train, sample_size, replace=False)
    
    X_train_sample_orig = X_train_orig[sample_indices]
    X_train_sample_scaled = X_train_scaled[sample_indices]
    
    all_results.extend(run_inference_on_split(X_train_sample_orig, X_train_sample_scaled, "Train"))
X_test_scaled = scaler.transform(np.nan_to_num(X_test_orig))
X_test_scaled[np.isnan(X_test_orig)] = np.nan

res_dir = Path("Inference_Results") / model_dir.name
res_dir.mkdir(parents=True, exist_ok=True)

# ------------------------- Inferenz-Loop über Maskierungslevel -------------------------
num_features = len(features)
for k in range(1, num_features):
    print(f"Processing Level {k}...")
    combos = list(itertools.combinations(range(num_features), k))
    
    if len(combos) > 50: combos_to_run = random.sample(combos, 50)
    else: combos_to_run = combos
    
    all_results = []
    for combo_indices in combos_to_run:
        X_target_scaled = X_test_scaled.copy()
        # ------------------------- Künstliche Maskierung -------------------------
        is_masked = np.zeros(X_target_scaled.shape, dtype=bool)
        for f_idx in combo_indices:
            mask_condition = ~np.isnan(X_target_scaled[:, f_idx])
            X_target_scaled[mask_condition, f_idx] = 0.0
            is_masked[mask_condition, f_idx] = True
        
        X_target_scaled = np.nan_to_num(X_target_scaled)
        
        # ------------------------- VAE Rekonstruktion -------------------------
        with torch.no_grad():
            recon, _, _ = vae(torch.from_numpy(X_target_scaled).float())
            X_recon_scaled = recon.numpy()
        
        X_recon_orig = scaler.inverse_transform(X_recon_scaled)
        
        # ------------------------- Ergebnisse sammeln -------------------------
        for f_idx in combo_indices:
            mask = is_masked[:, f_idx]
            if not mask.any(): continue
            
            o_vals = X_test_orig[mask, f_idx]
            i_vals = X_recon_orig[mask, f_idx]
            for o, i in zip(o_vals, i_vals):
                all_results.append({
                    "Feature": features[f_idx],
                    "Original": o,
                    "Imputed": i,
                    "Split": "Test",
                    "Masking_Level": k,
                    "Masked_Combination": str([features[i] for i in combo_indices])
                })
    
    # ------------------------- Speichern der Resultate -------------------------
    if all_results:
        df_res = pd.DataFrame(all_results)
        df_res.to_csv(res_dir / f"Imputation_Results_Run_{TARGET_RUN_INDEX:03d}_Level_{k}.csv", index=False, float_format='%.4f')
print("Done.")
